# 12 — Filters

Covers all five CASPI filter topologies: `StateVariable`, `Biquad`, `Ladder`, `DiodeLadder`, and `OnePole`.

**Sections:**
1. Filter response & attenuation — analytic |H(f)| for linear filters + PSD rolloff for all 5
2. Filtered saw wave — time domain, harmonic spectrum, audio for all 5 topologies
3. Interactive explorer — switch oscillator waveform / frequency / filter settings live

In [1]:
import caspy as cp
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch
from IPython.display import HTML, display, Audio
import ipywidgets as widgets
from ipykernel.comm import Comm
from scipy.signal import welch, spectrogram
import json
import io
import base64

SR = 44100

## 1. Filter response & attenuation

Linear filters (StateVariable, Biquad, OnePole) expose `frequency_response()` for analytic |H(f)|.
Nonlinear ladder filters have no closed-form transfer function; their effective rolloff is shown via noise PSD.

In [ ]:
FREQS = np.logspace(np.log10(20), np.log10(20000), 500, dtype=np.float64)
CUTOFF = 1000.0
Q_VAL  = 0.707

def analytic_response(filt, freq_grid):
    return np.array([filt.frequency_response(float(f)) for f in freq_grid], dtype=np.float64)

# Linear filters — analytic frequency response
lin_filters = {
    'SVF LP':     cp.StateVariable(SR, CUTOFF, Q_VAL, cp.FilterMode.LowPass),
    'SVF HP':     cp.StateVariable(SR, CUTOFF, Q_VAL, cp.FilterMode.HighPass),
    'SVF BP':     cp.StateVariable(SR, CUTOFF, Q_VAL, cp.FilterMode.BandPass),
    'Biquad LP':  cp.Biquad(SR, CUTOFF, Q_VAL, cp.FilterMode.LowPass),
    'Biquad HP':  cp.Biquad(SR, CUTOFF, Q_VAL, cp.FilterMode.HighPass),
    'OnePole LP': cp.OnePole(SR, CUTOFF, cp.FilterMode.LowPass),
    'OnePole HP': cp.OnePole(SR, CUTOFF, cp.FilterMode.HighPass),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, f in lin_filters.items():
    mag = analytic_response(f, FREQS)
    axes[0].semilogx(FREQS, mag, label=name, linewidth=1.2)
    axes[1].semilogx(FREQS, 20 * np.log10(np.maximum(mag, 1e-12)),
                     label=name, linewidth=1.2)

for ax in axes:
    ax.axvline(CUTOFF, color='grey', linestyle='--', linewidth=0.7, alpha=0.5)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_xlim(20, 20000)
    ax.grid(True, which='both', alpha=0.3)
axes[0].set_ylabel('|H(f)|')
axes[0].set_title(f'Linear magnitude — cutoff={CUTOFF} Hz, Q={Q_VAL}')
axes[0].set_ylim(0, 1.1)
axes[0].legend(fontsize=7, ncol=2)
axes[1].set_ylabel('|H(f)| (dB)')
axes[1].set_title('Magnitude response (dB)')
axes[1].set_ylim(-60, 3)
axes[1].legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

# All 5 topologies — rolloff from noise PSD
NOISE_LEN = SR * 4
noise_osc = cp.noise.NoiseOscillatorWhite(float(SR))
buf_in = noise_osc.render(NOISE_LEN)

all_filters = {
    'StateVariable LP': cp.StateVariable(SR, 1000.0, 0.707, cp.FilterMode.LowPass),
    'Biquad LP':        cp.Biquad(SR, 1000.0, 0.707, cp.FilterMode.LowPass),
    'Moog Ladder':      cp.Ladder(SR, 1000.0, 0.707),
    'Diode Ladder':     cp.DiodeLadder(SR, 1000.0, 0.707),
    'OnePole LP':       cp.OnePole(SR, 1000.0, cp.FilterMode.LowPass),
}

fig2, ax2 = plt.subplots(figsize=(11, 5))
f_in, psd_in = welch(buf_in, fs=SR, nperseg=4096)
ax2.semilogx(f_in[1:], 10 * np.log10(psd_in[1:]), color='black', linewidth=0.8,
             alpha=0.5, label='Input white noise')

for name, filt in all_filters.items():
    buf_out = filt.process_block(buf_in.copy())
    f_out, psd_out = welch(buf_out, fs=SR, nperseg=4096)
    ax2.semilogx(f_out[1:], 10 * np.log10(psd_out[1:]), linewidth=0.9, alpha=0.85,
                 label=name)

ax2.axvline(1000.0, color='grey', linestyle='--', linewidth=0.7, alpha=0.5)
ax2.set_xlabel('Frequency (Hz)')
ax2.set_ylabel('PSD (dB/Hz)')
ax2.set_title('White noise filtered at LP 1 kHz — PSD comparison (all 5 topologies)')
ax2.set_xlim(20, SR / 2)
ax2.legend(fontsize=8)
ax2.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

# Measure rolloff slopes in dB/decade
SLOPE_LO, SLOPE_HI = 2000.0, 8000.0
print(f'Rolloff slopes ({SLOPE_LO:.0f}–{SLOPE_HI:.0f} Hz):')
for name, filt in all_filters.items():
    buf_out = filt.process_block(buf_in.copy())
    f_o, psd_o = welch(buf_out, fs=SR, nperseg=4096)
    mask = (f_o >= SLOPE_LO) & (f_o <= SLOPE_HI)
    if mask.sum() > 5:
        log_f = np.log10(f_o[mask])
        log_p = np.log10(psd_o[mask] + 1e-30)
        coeffs = np.polyfit(log_f, log_p, 1)
        slope_db = coeffs[0] * 10.0
        pole_est = -slope_db / 20.0
        print(f'  {name:20s}  {slope_db:+6.1f} dB/decade  (~{pole_est:.1f} pole)')

## 2. Filtered saw wave — time domain + harmonic spectrum + audio

A saw wave is rich in harmonics — ideal for comparing how each filter topology shapes the spectrum.

In [ ]:
SAW_FREQ = 220.0
N_SAW    = int(SR * 0.2)  # 200 ms

osc = cp.oscillators.BlepOscillator()
osc.set_shape(cp.oscillators.WaveShape.Saw)
osc.set_sample_rate(SR)
osc.set_frequency(SAW_FREQ)
buf_saw = osc.render(N_SAW).astype(np.float32)

saw_filters = {
    'Input (saw)':    None,
    'SVF LP 500':     cp.StateVariable(SR, 500.0, 0.707, cp.FilterMode.LowPass),
    'Biquad LP 500':  cp.Biquad(SR, 500.0, 0.707, cp.FilterMode.LowPass),
    'Moog Ladder':    cp.Ladder(SR, 500.0, 0.707),
    'Diode Ladder':   cp.DiodeLadder(SR, 500.0, 0.707),
    'OnePole LP 500': cp.OnePole(SR, 500.0, cp.FilterMode.LowPass),
}

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for idx, (name, filt) in enumerate(saw_filters.items()):
    if filt is None:
        buf_out = buf_saw
        color = 'black'
    else:
        buf_out = filt.process_block(buf_saw.copy())
        color = None
    t_ms = np.arange(len(buf_out)) / SR * 1000
    axes[idx].plot(t_ms[:200], buf_out[:200], linewidth=0.8, color=color)
    axes[idx].set_title(name, fontsize=9)
    axes[idx].set_xlabel('Time (ms)')
    axes[idx].set_ylabel('Amplitude')
    axes[idx].set_ylim(-1.2, 1.2)
    axes[idx].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Harmonic spectrum overlay
def harmonic_spectrum(buf, n_fft=8192):
    window = np.hanning(min(len(buf), n_fft))
    if len(buf) < n_fft:
        buf = np.pad(buf, (0, n_fft - len(buf)))
    spectrum = np.fft.rfft(buf[:n_fft] * window)
    freqs = np.fft.rfftfreq(n_fft, d=1.0 / SR)
    return freqs, np.abs(spectrum)

freqs, mag_in = harmonic_spectrum(buf_saw)
fig2, ax_spec = plt.subplots(figsize=(12, 5))
ax_spec.semilogx(freqs[1:], 20 * np.log10(mag_in[1:] / np.max(mag_in)),
                 color='black', alpha=0.4, linewidth=0.8, label='Input saw')

for label, filt in saw_filters.items():
    if filt is None:
        continue
    buf_f = filt.process_block(buf_saw.copy())
    _, mag_f = harmonic_spectrum(buf_f)
    ax_spec.semilogx(freqs[1:], 20 * np.log10(mag_f[1:] / np.max(mag_in)),
                     linewidth=1.0, label=label)

ax_spec.axvline(500.0, color='grey', linestyle='--', linewidth=0.7, alpha=0.5)
ax_spec.set_xlabel('Frequency (Hz)')
ax_spec.set_ylabel('Magnitude (dB, norm to input peak)')
ax_spec.set_title('Harmonic spectrum — 220 Hz saw through LP at 500 Hz')
ax_spec.set_xlim(20, 10000)
ax_spec.set_ylim(-70, 5)
ax_spec.legend(fontsize=8)
ax_spec.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

# Audio playback
for name, filt in saw_filters.items():
    if filt is None:
        display(Audio(buf_saw * 0.3, rate=SR, normalize=False))
    else:
        display(Audio(filt.process_block(buf_saw.copy()) * 0.3, rate=SR, normalize=False))

In [ ]:
print(widgets.VBox)

<class 'ipywidgets.widgets.widget_box.VBox'>


## 3. Interactive filter explorer

Pick an oscillator waveform, set its frequency, choose a filter topology / mode / cutoff / Q, and hear the result.

In [ ]:
# Assume SR, cp already defined in notebook context

w_wave = widgets.Dropdown(
    options=['Sine', 'Saw', 'Square', 'Triangle'],
    value='Saw',
    description='Wave'
)

w_freq = widgets.FloatSlider(
    value=220.0,
    min=40.0,
    max=2000.0,
    step=1.0,
    description='Freq (Hz)',
    continuous_update=False
)

w_topo = widgets.Dropdown(
    options=['StateVariable', 'Biquad', 'Ladder', 'DiodeLadder', 'OnePole'],
    value='StateVariable',
    description='Topology'
)

w_mode = widgets.Dropdown(
    options=['LowPass', 'HighPass', 'BandPass', 'Notch', 'Peak'],
    value='LowPass',
    description='Mode'
)

w_cutoff = widgets.FloatLogSlider(
    value=1000.0,
    base=10,
    min=1.3,  # 20 Hz
    max=4.3,  # 20000 Hz
    step=0.02,
    description='Cutoff (Hz)',
    continuous_update=False
)

w_q = widgets.FloatSlider(
    value=0.707,
    min=0.1,
    max=20.0,
    step=0.05,
    description='Q',
    continuous_update=False
)

output_area = widgets.Output()

def update_explorer(change=None):
    """Render filter output: waveform, PSD, spectrogram, audio."""
    with output_area:
        output_area.clear_output(wait=True)

        try:
            N = SR * 1  # 1 second
            
            osc = cp.oscillators.BlepOscillator()
            osc.set_sample_rate(SR)
            osc.set_frequency(w_freq.value)
            
            wave_map = {
                'Sine': cp.oscillators.WaveShape.Sine,
                'Saw': cp.oscillators.WaveShape.Saw,
                'Square': cp.oscillators.WaveShape.Square,
                'Triangle': cp.oscillators.WaveShape.Triangle
            }
            osc.set_shape(wave_map[w_wave.value])
            buf_in = osc.render(N).astype(np.float32)
            
            fb = cp.FilterBank(SR)
            fb.set_cutoff(w_cutoff.value)
            fb.set_q(w_q.value)
            
            mode_map = {
                'LowPass': cp.FilterMode.LowPass,
                'HighPass': cp.FilterMode.HighPass,
                'BandPass': cp.FilterMode.BandPass,
                'Notch': cp.FilterMode.Notch,
                'Peak': cp.FilterMode.Peak
            }
            fb.set_mode(mode_map[w_mode.value])
            
            topo_idx = {
                'StateVariable': 0, 'Biquad': 1, 'Ladder': 2,
                'DiodeLadder': 3, 'OnePole': 4
            }
            fb.set_active_index(topo_idx[w_topo.value])
            
            buf_out = fb.process_block(buf_in.copy())
            f_psd, psd = welch(buf_out, fs=SR, nperseg=2048)
            f_spec, t_spec, Sxx = spectrogram(buf_out, fs=SR, nperseg=512, noverlap=256)
            
            fig, axes = plt.subplots(3, 1, figsize=(12, 9))
            
            # Waveform
            t_ms = np.arange(min(4410, N)) / SR * 1000
            axes[0].plot(t_ms, buf_out[:4410], linewidth=0.6, color='#1f77b4')
            axes[0].set_xlabel('Time (ms)')
            axes[0].set_ylabel('Amplitude')
            axes[0].set_title(f"Waveform: {w_wave.value} @ {w_freq.value:.0f} Hz")
            axes[0].set_ylim(-1.2, 1.2)
            axes[0].grid(True, alpha=0.3)
            
            # PSD
            axes[1].semilogx(f_psd[1:], 10 * np.log10(psd[1:] + 1e-30), linewidth=0.7, color='#ff7f0e')
            axes[1].axvline(w_cutoff.value, color='red', linestyle='--', linewidth=1, alpha=0.7)
            axes[1].set_xlabel('Frequency (Hz)')
            axes[1].set_ylabel('PSD (dB/Hz)')
            title = f"{w_topo.value} {w_mode.value}"
            if w_topo.value != 'OnePole':
                title += f", Q={w_q.value:.2f}"
            axes[1].set_title(title + f" | Cutoff: {w_cutoff.value:.0f} Hz")
            axes[1].set_xlim(20, SR / 2)
            axes[1].set_ylim(-60, 10)
            axes[1].grid(True, which='both', alpha=0.3)
            
            # Spectrogram
            Sxx_db = 10 * np.log10(Sxx + 1e-30)
            im = axes[2].pcolormesh(t_spec, f_spec, Sxx_db, shading='gouraud', cmap='viridis', vmin=-80, vmax=0)
            axes[2].set_ylabel('Frequency (Hz)')
            axes[2].set_xlabel('Time (s)')
            axes[2].set_title('Spectrogram')
            axes[2].set_ylim(0, min(5000, SR / 2))
            plt.colorbar(im, ax=axes[2], label='dB')
            
            plt.tight_layout()
            plt.show()
            
            display(Audio(buf_out * 0.3, rate=SR, normalize=False))   
        except Exception:
            traceback.print_exc()


# Attach observers to all controls
for widget in [w_wave, w_freq, w_topo, w_mode, w_cutoff, w_q]:
    widget.observe(update_explorer, names='value')

# Render UI
display(widgets.VBox([
    widgets.HBox([w_wave, w_freq]),
    widgets.HBox([w_topo, w_mode]),
    widgets.HBox([w_cutoff, w_q]),
    output_area
]))


# Initial render
update_explorer()

Dropdown(description='Wave', index=1, options=('Sine', 'Saw', 'Square', 'Triangle'), value='Saw')

FloatSlider(value=220.0, continuous_update=False, description='Freq (Hz)', max=2000.0, min=40.0, step=1.0)

Dropdown(description='Topology', options=('StateVariable', 'Biquad', 'Ladder', 'DiodeLadder', 'OnePole'), valu…

Dropdown(description='Mode', options=('LowPass', 'HighPass', 'BandPass', 'Notch', 'Peak'), value='LowPass')

In [ ]:

import ipywidgets
from IPython.display import display

display(ipywidgets.IntSlider())


IntSlider(value=0)